In [231]:
from pathlib import Path
notebook_directory = Path.cwd()

import numpy as np
import pandas as pd
import xlwings as xw

In [232]:
sector = 'Consumer Staples' 

prices_csv = 'sectors'

#start_date = pd.Timestamp("2026-03-25")
#end_date   = pd.Timestamp("2026-06-14")

start_date = pd.Timestamp("2026-06-25")
end_date   = pd.Timestamp("2026-08-25")

output_price_filename = sector 

In [233]:
ishare_fee_diff = .003

In [234]:
prices_path = (
    notebook_directory.parent #.parent
    / "backtesting"
    / "historical prices"
    / f"{prices_csv}.csv"
)
prices_df = pd.read_csv(prices_path, parse_dates=["date"])

'''
db_path = (
    notebook_directory.parent
    / "spreadsheets"
    / "2026 Fin Inst Database.xlsx"
)
wb = xw.Book(db_path)
ws = wb.sheets["Scalar Inputs Table"]
db_df = ws.tables["scalar_inputs_table"].range.options(pd.DataFrame, header=1, index=False).value
'''


scalar_path = (
    notebook_directory.parent
    / "trading"
    / "spreadsheets"
    / "2026 Group Trading Inputs.xlsm"
)
wb = xw.Book(scalar_path)
ws = wb.sheets["SCALAR OUTPUTS"]
scalar_df = ws.range("A1:I29").options(pd.DataFrame, header=1, index=False).value

In [235]:
scalar_df = scalar_df[scalar_df['sector'] == sector].copy()

In [236]:
symbols = scalar_df['symbol'].to_list()

In [237]:
#previous_month = (pd.Timestamp.today() - pd.DateOffset(months=1)).to_period("M")
#prices_df = prices_df.loc[prices_df["date"].dt.to_period("M").eq(previous_month)].copy()

prices_df["date"] = pd.to_datetime(prices_df["date"], errors="coerce")
prices_df = prices_df.loc[prices_df["date"].between(start_date, end_date)].copy()

prices_df = prices_df[['date'] + symbols]

In [238]:
for sym in symbols:
    scalar = scalar_df.loc[scalar_df['symbol'] == sym, 'slope'].to_list()[0]
    prices_df[f"{sym} scalar"] = scalar

In [239]:
print(scalar_df)

    group            sector symbol my_platform anchor intercept     slope  \
0  Sector  Consumer Staples   FSTA        IBKR    VDC   -1.9182  4.329231   
1  Sector  Consumer Staples    VDC        IBKR    VDC         0  1.000000   
2  Sector  Consumer Staples    XLP        IBKR    VDC   33.8394  2.325458   

  intercept no div  slope no div  
0          -1.8607      4.329231  
1                0      1.000000  
2          33.9508      2.325458  


In [240]:
unit_price_cols_div = []
unit_price_cols_no_div = []

for sym in symbols:
    col_name = f'{sym} unit price div'
    const = scalar_df.loc[scalar_df['symbol'].eq(sym), 'intercept'].to_list()[0]
    prices_df[col_name] = prices_df[sym] * prices_df[f"{sym} scalar"] + float(const)
    unit_price_cols_div.append(col_name)

    col_name = f'{sym} unit price no div'
    const = scalar_df.loc[scalar_df['symbol'].eq(sym), 'intercept no div'].to_list()[0]
    prices_df[col_name] = prices_df[sym] * prices_df[f"{sym} scalar"] + float(const)
    unit_price_cols_no_div.append(col_name)

In [241]:
'''
start_date = prices_df['date'].iloc[0]
prices_df["days"] = (prices_df["date"] - start_date).dt.days
'''

'\nstart_date = prices_df[\'date\'].iloc[0]\nprices_df["days"] = (prices_df["date"] - start_date).dt.days\n'

In [242]:
'''
ishare_daily_log_rate = np.log(ishare_fee_diff) / 365
prices_df["ishare_scalar"] = 1 #np.exp(ishare_daily_log_rate * prices_df["days"])
'''

'\nishare_daily_log_rate = np.log(ishare_fee_diff) / 365\nprices_df["ishare_scalar"] = 1 #np.exp(ishare_daily_log_rate * prices_df["days"])\n'

In [243]:
'''
ishare_symbols = [symbol for symbol in symbols if symbol.lower().startswith("i")]
for sym in ishare_symbols:
    col_name = f'{sym} unit price'
    new_col_name = f'{sym}* unit price'
    prices_df[new_col_name] = prices_df[col_name] * prices_df["ishare_scalar"]
    unit_price_cols[unit_price_cols.index(col_name)] = new_col_name
'''


'\nishare_symbols = [symbol for symbol in symbols if symbol.lower().startswith("i")]\nfor sym in ishare_symbols:\n    col_name = f\'{sym} unit price\'\n    new_col_name = f\'{sym}* unit price\'\n    prices_df[new_col_name] = prices_df[col_name] * prices_df["ishare_scalar"]\n    unit_price_cols[unit_price_cols.index(col_name)] = new_col_name\n'

In [244]:
prices_df["max_unit_price"] = prices_df[unit_price_cols_div].max(axis=1)
prices_df["min_unit_price"] = prices_df[unit_price_cols_div].min(axis=1)
prices_df['pct_diff'] = np.log(prices_df['max_unit_price'] / prices_df['min_unit_price'])
prices_df["max_sym"] = prices_df[unit_price_cols_div].idxmax(axis=1)
prices_df["min_sym"] = prices_df[unit_price_cols_div].idxmin(axis=1)

In [245]:
prices_df["max_unit_price_no"] = prices_df[unit_price_cols_no_div].max(axis=1)
prices_df["min_unit_price_no"] = prices_df[unit_price_cols_no_div].min(axis=1)
prices_df['pct_diff_no'] = np.log(prices_df['max_unit_price_no'] / prices_df['min_unit_price_no'])
prices_df["max_sym_no"] = prices_df[unit_price_cols_no_div].idxmax(axis=1)
prices_df["min_sym_no"] = prices_df[unit_price_cols_no_div].idxmin(axis=1)

In [246]:
prices_path = (
    notebook_directory.parent #.parent
    / "backtesting"
    / "backtests"
    / f"{output_price_filename}.csv"
)

prices_df.to_csv(prices_path, index=False)